In [1]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS
from linearmodels.datasets import wage_panel

from pymargins import Margins, pairwise

raw = wage_panel.load()
df = raw.set_index(["nr", "year"]).copy()
print(df[["lwage", "educ", "exper", "expersq",
          "union", "married"]].describe().round(2))

         lwage     educ    exper  expersq    union  married
count  4360.00  4360.00  4360.00  4360.00  4360.00  4360.00
mean      1.65    11.77     6.51    50.42     0.24     0.44
std       0.53     1.75     2.83    40.78     0.43     0.50
min      -3.58     3.00     0.00     0.00     0.00     0.00
25%       1.35    11.00     4.00    16.00     0.00     0.00
50%       1.67    12.00     6.00    36.00     0.00     0.00
75%       1.99    12.00     9.00    81.00     0.00     1.00
max       4.05    16.00    18.00   324.00     1.00     1.00


In [2]:
fe = PanelOLS(
    df["lwage"],
    df[["exper", "expersq", "union", "married"]],
    entity_effects=True,
).fit(cov_type="clustered", cluster_entity=True)
print(fe.summary.tables[1])

                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
exper          0.1168     0.0107     10.917     0.0000      0.0959      0.1378
expersq       -0.0043     0.0007    -6.2744     0.0000     -0.0056     -0.0030
union          0.0821     0.0228     3.5994     0.0003      0.0374      0.1268
married        0.0453     0.0210     2.1589     0.0309      0.0042      0.0864


In [3]:
m = Margins.linear_scale(fe, at="overall")
scen, w = pairwise("union", [1, 0])
print(m.contrasts(scenarios=scen, contrasts=w).summary())

             Margins Result (delta, level=0.95)            
         estimate  std err       z  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
union=1    0.0821   0.0228  3.5994  0.000    0.0374, 0.1268

n = 4360
κ: 0.000
Delta-vs-sim disagreement: 6.569%


In [4]:
m_log = Margins.log_scale(fe, at="overall")
print(m_log.contrasts(scenarios=scen, contrasts=w).summary())

             Margins Result (delta, level=0.95)            
         estimate  std err       z  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
union=1    1.1455   0.0371  3.6651  0.000    1.0652, 1.2318

n = 4360
Note: std err is on the inference scale; estimate and CI are on the reporting scale.
κ: 0.064
Delta-vs-sim disagreement: 0.199%


In [5]:
m_sim = Margins.log_scale(
    fe, at="overall",
    method="simulation", n_sim=2000, rng_seed=0,
)
print(m_sim.contrasts(scenarios=scen, contrasts=w).summary())

           Margins Result (simulation, level=0.95)            
         estimate  std err  statistic  P>|z|  [95% Conf. Int.]
--------------------------------------------------------------
union=1    1.1455   0.0364     0.1358  0.000    1.0671, 1.2296

n = 4360
Note: std err is on the inference scale; estimate and CI are on the reporting scale.
κ: 0.064
